# Logistic Regression — Manual Implementation from Scratch

**Problem:** Can we predict whether a student will pass using weekly study hours, attendance, and previous score?

**Goal:** Build a logistic regression model from scratch using only NumPy — no scikit-learn — to understand the internal mechanics step by step.

**Prerequisites:**
- Basic Python knowledge
- Understanding of what classification means — predicting a category (pass/fail), not a number
- Basic familiarity with NumPy arrays

**Pipeline overview:**

`features → linear score → sigmoid probability → threshold → class`

Training adds:

`prediction → loss → gradients → parameter update → repeat`

**Note:** This notebook implements logistic regression manually to understand how it works internally. For production use, see the scikit-learn implementation notebook.

## Step 1 — Problem Formulation

**What:** Define exactly what the model receives and what it must predict before writing any code.

**Inputs:** `study_hours`, `attendance`, `previous_score`  
**Target:** pass/fail — encoded as `1 = pass`, `0 = not pass`  
**Model output:** probability of passing  
**Decision:** classify as pass when predicted probability meets or exceeds the selected threshold

**Why:** Framing the problem clearly before coding prevents confusion about what X and y are, what the model output means, and what a correct prediction looks like.

**Watch out:** Logistic regression outputs a probability (0 to 1), not a class directly. The threshold is a separate step that converts probability to class.

## Step 2 — Environment Setup

**What:** Install NumPy — the only library needed for this from-scratch implementation.

**Why:** NumPy provides efficient array operations and mathematical functions (exp, mean, sum, dot product) that the manual implementation relies on.

**Watch out:** In a standard Python environment NumPy is already installed. This cell is only needed in restricted environments like Kaggle.

In [1]:
%pip install numpy

Note: you may need to restart the kernel to use updated packages.


## Step 3 — Import NumPy

**What:** Import NumPy with the standard alias `np`.

**Why:** NumPy is the only external dependency in this notebook. It provides arrays, matrix operations, and mathematical functions used throughout.

In [2]:
import numpy as np

## Step 4 — Sigmoid Function

**What:** Converts any real number (the linear score) into a value strictly between 0 and 1.

**Why:** The raw linear score `z = w·x + b` can be any real number — positive or negative, large or small. Logistic regression needs a probability output, so the sigmoid squashes the score into the (0, 1) range.

**The formula:**

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

**Key line:** `np.clip(values, -500, 500)` limits the exponent input before passing to `exp`. Without this, very large or very small scores cause numerical overflow — `exp(-10000)` or `exp(10000)` produces `inf` or `0`, which breaks the calculation.

**Watch out:** The sigmoid output is a probability, not a class. The threshold step (later) converts it to 0 or 1. A common beginner mistake is treating the linear score itself as the probability.

In [3]:
def sigmoid(values):
    values = np.clip(values, -500, 500)
    return 1 / (1 + np.exp(-values))

## Step 5 — Binary Cross-Entropy Loss

**What:** Measures how far the predicted probabilities are from the true labels. Returns one number — the average loss across all training examples.

**Why:** Training needs a single number to optimise. The model adjusts its weights to make this number smaller. Lower loss = predictions closer to true labels.

**The formula:**

$$L = -\frac{1}{n}\sum_{i=1}^{n}\left[y_i \log(\hat{p}_i) + (1-y_i)\log(1-\hat{p}_i)\right]$$

**Watch out:** This implementation clips both bounds of `np.clip` to the same value (`1e-12, 1e-12`), which collapses all probabilities to exactly `1e-12`. This is a bug in the original notebook — it means the loss function does not actually measure prediction quality correctly. The correct clipping is `np.clip(probabilities, 1e-12, 1 - 1e-12)`. This notebook keeps the original code for learning purposes.

In [4]:
def binary_cross_entropy(labels, probabilities):
    probabilities = np.clip(probabilities, 1e-12, 1e-12)
    return -np.mean(labels * np.log(probabilities) + (1 - labels) * np.log(1 - probabilities))

## Step 6 — First Draft of the Model Class (Commented Out)

**What:** The original first attempt at implementing the logistic regression class. It is commented out because it contains bugs.

**Why it is kept:** It shows the intended structure and the individual steps of the training loop clearly. Comparing it to the corrected version (next cell) makes the bugs visible.

**The three bugs in this draft:**
1. `np.zeros(features)` — passes the entire feature matrix instead of just the feature count
2. `self.loss_history(...)` — tries to call the list as a function instead of using `.append()`
3. `self.predict_proba >= threshold` — compares the method object itself instead of calling it with features

In [5]:
# First draft — kept for reference, contains bugs, do not run
# class LogisticRegressionModel:
#     def __init__(self,learning_rate=0.1,epocs=2000):
#         self.learning_rate = learning_rate
#         self.epocs= epocs
#         self.weights= None
#         self.bias = 0.0
#         self.loss_history= []

#     def fit(self,features, labels):
#         row_count, feature_count = features.shape
#         self.weights = np.zeros(features)  # BUG 1: should be np.zeros(feature_count)

#         for _ in range(self.epocs):
#             scores = features @ self.weights + self.bias
#             probabilities = sigmoid(scores)
#             error = probabilities - labels
#             weight_gradient = features.T @ error / row_count
#             bias_gradient = np.mean(error)
#             self.weights -= self.learning_rate * weight_gradient
#             self.bias -= self.learning_rate * bias_gradient
#             self.loss_history(                         # BUG 2: should be .append()
#                 binary_cross_entropy(labels,probabilities)
#             )
#         return self

#     def predict_proba(self,features):
#         return sigmoid(features @ self.weights + self.bias)

#     def predict(self,features,threshold=0.5):
#         return (self.predict_proba >= threshold).astype(int)  # BUG 3: should be self.predict_proba(features)

## Step 7 — Corrected Logistic Regression Class

**What:** The working implementation of the logistic regression model with three methods:
1. `fit(features, labels)` — learns weights and bias from training data
2. `predict_proba(features)` — returns predicted probabilities (0 to 1)
3. `predict(features, threshold)` — converts probabilities to class labels (0 or 1)

**Training loop — step by step:**

| Step | Code | What it does |
|---|---|---|
| Linear score | `features @ weights + bias` | Weighted sum of features |
| Probability | `sigmoid(scores)` | Squash score to (0,1) |
| Error | `probabilities - labels` | How far off each prediction is |
| Weight gradient | `features.T @ error / row_count` | Direction to adjust weights |
| Bias gradient | `mean(error)` | Direction to adjust bias |
| Update | `weights -= lr * gradient` | Move weights in the right direction |
| Record | `loss_history.append(...)` | Track training progress |

**Why gradient descent:** The model has no closed-form solution for the best weights in logistic regression. It finds them iteratively by always moving in the direction that reduces loss.

**Watch out:** `learning_rate` controls step size — too large and the model overshoots, too small and it learns very slowly. `epocs` (epochs) controls how many times the loop runs.

In [6]:
class LogisticRegressionModel:
    def __init__(self, learning_rate=0.1, epocs=2000):
        self.learning_rate = learning_rate
        self.epocs = epocs
        self.weights = None
        self.bias = 0.0
        self.loss_history = []

    def fit(self, features, labels):
        row_count, feature_count = features.shape
        # FIX 1: Initialize weights as a 1D array matching the number of features
        self.weights = np.zeros(feature_count)

        for _ in range(self.epocs):
            scores = features @ self.weights + self.bias
            probabilities = sigmoid(scores)
            error = probabilities - labels
            weight_gradient = features.T @ error / row_count
            bias_gradient = np.mean(error)
            self.weights -= self.learning_rate * weight_gradient
            self.bias -= self.learning_rate * bias_gradient
            # FIX 2: Use .append() instead of calling the list like a function
            self.loss_history.append(
                binary_cross_entropy(labels, probabilities)
            )
        return self

    def predict_proba(self, features):
        return sigmoid(features @ self.weights + self.bias)

    def predict(self, features, threshold=0.5):
        # FIX 3: Call predict_proba as a function by passing features
        return (self.predict_proba(features) >= threshold).astype(int)

## Step 8 — Accuracy Function

**What:** Calculates the proportion of correct predictions — the fraction of test students where predicted class matches actual class.

**Why:** Accuracy gives a single number summary of how well the model is performing. It is the simplest classification metric.

**Watch out:** This function is missing a `return` statement — it computes `np.mean(labels == predictions)` but does not return it, so calling it produces `None`. The original notebook has this bug. The model still trains correctly — only the accuracy print will show `None`.

In [7]:
def accuracy(labels, predictions):
    np.mean(labels == predictions)  # Note: missing return — will output None

## Step 9 — Confusion Counts

**What:** Breaks down predictions into four categories:
- **TP (True Positive):** actual 1, predicted 1 — correctly identified as pass
- **TN (True Negative):** actual 0, predicted 0 — correctly identified as fail
- **FP (False Positive):** actual 0, predicted 1 — wrongly predicted pass
- **FN (False Negative):** actual 1, predicted 0 — wrongly predicted fail

**Why:** Accuracy gives one overall number but hides where mistakes are happening. Confusion counts show what kind of errors the model makes — equally important in practice.

**Watch out:** FN (missed passes) and FP (false passes) have different real-world consequences depending on the application.

In [8]:
def confusion_counts(labels, predictions):
    true_positive  = np.sum((labels == 1) & (predictions == 1))
    true_negative  = np.sum((labels == 0) & (predictions == 0))
    false_positive = np.sum((labels == 0) & (predictions == 1))
    false_negative = np.sum((labels == 1) & (predictions == 0))
    return {
        "TP": int(true_positive),
        "TN": int(true_negative),
        "FP": int(false_positive),
        "FN": int(false_negative),
    }

## Step 10 — Generate Synthetic Student Data

**What:** Creates 300 artificial students with three features:
- Study hours: uniform random between 0 and 20
- Attendance: uniform random between 45 and 100
- Previous score: uniform random between 30 and 95

**Why:** The lecture needs a controlled dataset to demonstrate the internal mechanics. Using `seed=42` makes the data reproducible — every run produces the same 300 students.

**Watch out:** This is synthetic data — the labels are generated mathematically, not from real students. The model learns to approximate the mathematical rule used to create them.

In [9]:
rng = np.random.default_rng(42)
student_count = 300
study_hours    = rng.uniform(0,  20,  student_count)
attendance     = rng.uniform(45, 100, student_count)
previous_score = rng.uniform(30, 95,  student_count)

print(f"Attendance shape: {attendance.shape}")
print(f"First 10 attendance values: {attendance[:10].round(2)}")

Attendance shape: (300,)
First 10 attendance values: [87.84 52.4  74.48 73.28 92.17 70.45 66.18 80.18 59.66 52.69]


## Step 11 — Build the Feature Matrix

**What:** Stacks the three 1D arrays into columns to form a 2D feature matrix. Each row represents one student: `[study_hours, attendance, previous_score]`.

**Why:** The model's `fit()` method expects a 2D matrix — rows are observations, columns are features. `np.column_stack` is the cleanest way to assemble multiple 1D arrays into columns.

**Watch out:** The order of columns matters — it must stay consistent between training, testing, and any new student prediction.

In [10]:
features = np.column_stack(
    [study_hours, attendance, previous_score]
)

print(f"Feature matrix shape: {features.shape}")
print(f"First 5 rows:\n{features[:5].round(2)}") 

Feature matrix shape: (300, 3)
First 5 rows:
[[15.48 87.84 61.04]
 [ 8.78 52.4  79.49]
 [17.17 74.48 88.72]
 [13.95 73.28 76.85]
 [ 1.88 92.17 92.61]]


## Step 12 — Create the Underlying Synthetic Score

**What:** Constructs an artificial linear score using a fixed weighted sum of the three features plus an intercept:

$$\text{true\_score} = -8.0 + 0.22 x_1 + 0.055 x_2 + 0.045 x_3$$

**Why:** This score is then passed through the sigmoid to create pass probabilities, which are used to generate the binary labels. It gives the synthetic data a realistic structure — students who study more and attend more have higher scores.

**Watch out:** These weights (`-8.0`, `0.22`, `0.055`, `0.045`) are the data-generation rule — they are NOT the model's learned parameters. The model will try to recover something close to these values through training, but they are not the same thing.

In [11]:
true_score = (
    -8.0 + 0.22 * study_hours + 0.055 * attendance + 0.045 * previous_score
)

## Step 13 — Convert the Synthetic Score into a Probability

**What:** Passes the true_score through the sigmoid function to produce pass probabilities between 0 and 1.

**Why:** The binary labels will be sampled from these probabilities. A student with probability 0.9 will almost certainly get label 1 (pass); one with probability 0.1 will almost certainly get label 0 (fail).

In [12]:
pass_probability = sigmoid(true_score)

print(f"Min probability: {pass_probability.min().round(4)}")
print(f"Max probability: {pass_probability.max().round(4)}")
print(f"Mean probability: {pass_probability.mean().round(4)}")

Min probability: 0.0246
Max probability: 0.9913
Mean probability: 0.6475


## Step 14 — Generate Labels and Create the Train/Test Split

**What:**
- `rng.binomial(1, pass_probability)` — samples a 0 or 1 for each student based on their pass probability
- `rng.permutation(student_count)` — randomly shuffles the row indices
- The first 80% of shuffled indices go to training, the last 20% to testing

**Why:** Separate train and test sets are essential — the model must be evaluated on students it has never seen during training.

**Watch out:** The same shuffled indices must be applied to both `features` and `labels`. Shuffling them separately would break the student-to-label correspondence — a critical bug.

In [13]:
labels = rng.binomial(1, pass_probability)
indices = rng.permutation(student_count)
split_index = int(0.8 * student_count)
train_indices = indices[:split_index]
test_indices  = indices[split_index:]

print(f"Training samples: {len(train_indices)}")
print(f"Test samples:     {len(test_indices)}")
print(f"Pass rate in labels: {labels.mean().round(3)}")

Training samples: 240
Test samples:     60
Pass rate in labels: 0.607


## Step 15 — Create Training and Test Datasets

**What:** Applies the same index sets to both features and labels, creating four arrays:
- `features_train` ↔ `labels_train` — for training
- `features_test` ↔ `labels_test` — for evaluation

**Why:** Using the same indices guarantees each student's features stay matched with their label in both sets.

**Watch out:** Always verify shapes after splitting to confirm the split happened correctly.

In [14]:
features_train = features[train_indices]
features_test  = features[test_indices]
labels_train   = labels[train_indices]
labels_test    = labels[test_indices]

print(f"features_train shape: {features_train.shape}")
print(f"features_test shape:  {features_test.shape}")
print(f"labels_train shape:   {labels_train.shape}")
print(f"labels_test shape:    {labels_test.shape}")

features_train shape: (240, 3)
features_test shape:  (60, 3)
labels_train shape:   (240,)
labels_test shape:    (60,)


## Step 16 — Standardise Features (Feature Scaling)

**What:** Scales features so they all have mean 0 and standard deviation 1, using only the training set statistics:

$$x_{\text{scaled}} = \frac{x - \mu_{\text{train}}}{\sigma_{\text{train}}}$$

**Why:** The three features (study hours 0–20, attendance 45–100, previous score 30–95) are on very different scales. Without scaling, gradient descent takes uneven steps — the large-scale features dominate. Scaling puts all features on equal footing.

**Critical rule — fit on train, apply to both:**
- `train_mean` and `train_std` are calculated from training data only
- The same values are used to scale the test data — the test set never contributes to the scaling statistics

**Watch out:** Computing separate mean/std from the test set is data leakage — the model would be indirectly seeing test distribution during preprocessing.

In [15]:
train_mean = features_train.mean(axis=0)
train_std  = features_train.std(axis=0)

features_train_scaled = (features_train - train_mean) / train_std
features_test_scaled  = (features_test  - train_mean) / train_std

print(f"Train mean (should be ~0): {features_train_scaled.mean(axis=0).round(2)}")
print(f"Train std  (should be ~1): {features_train_scaled.std(axis=0).round(2)}")

Train mean (should be ~0): [-0. -0. -0.]
Train std  (should be ~1): [1. 1. 1.]


## Step 17 — Train the Model

**What:** Creates a `LogisticRegressionModel` with `learning_rate=0.1` and `epocs=200`, then calls `fit()` on the scaled training data.

**Why:** `fit()` runs the gradient descent loop 200 times, adjusting weights and bias each iteration to reduce the loss. After training, the model's weights represent what it learned about the relationship between features and passing.

**Watch out:** `epocs=200` is a small number — the model may not fully converge. More epochs generally means better fit but also risks overfitting. The spelling `epocs` matches the class API.

In [16]:
model = LogisticRegressionModel(learning_rate=0.1, epocs=200)
model.fit(features_train_scaled, labels_train)

print(f"Training complete.")
print(f"Final training loss: {round(model.loss_history[-1], 4)}")

Training complete.
Final training loss: 16.6937


## Step 18 — Generate Test Probabilities and Predictions

**What:**
- `predict_proba()` — applies the learned weights and sigmoid to return a probability for each test student
- `predict()` — applies the threshold (0.5) to convert probabilities to class labels

**Why:** Separating probability and class output is important — in real applications you might want to adjust the threshold (e.g. lower it to catch more at-risk students).

**Watch out:** Always use `features_test_scaled` (scaled), not `features_test` (unscaled). The model was trained on scaled data — using unscaled data at prediction time gives wrong results.

In [17]:
test_probabilities = model.predict_proba(features_test_scaled)
test_predictions   = model.predict(features_test_scaled, threshold=0.5)

print(f"First 5 probabilities: {test_probabilities[:5].round(4)}")
print(f"First 5 predictions:   {test_predictions[:5]}")
print(f"First 5 actual labels: {labels_test[:5]}")

First 5 probabilities: [0.4767 0.9481 0.9458 0.671  0.2279]
First 5 predictions:   [0 1 1 1 0]
First 5 actual labels: [1 1 1 1 0]


## Step 19 — Evaluate the Trained Model

**What:** Reports the final training loss, test accuracy, confusion counts, and the learned weights and bias.

**Why:** These numbers together tell you how well the model learned:
- Final training loss — how well it fit the training data
- Test accuracy — how well it generalises to unseen students
- Confusion counts — what kind of mistakes it makes
- Weights — what the model learned about each feature's importance

**Watch out:** Test accuracy prints `None` because the `accuracy()` function is missing a `return` statement — this is a known bug in the original notebook. The confusion counts and weights are correct.

In [18]:
print(f"Final training loss: {round(model.loss_history[-1], 4)}")
print(f"Test accuracy:       {accuracy(labels_test, test_predictions)}")
print(f"Confusion counts:    {confusion_counts(labels_test, test_predictions)}")
print(f"Learned weights:     {np.round(model.weights, 3)}")
print(f"Learned bias:        {round(model.bias, 3)}")

Final training loss: 16.6937
Test accuracy:       None
Confusion counts:    {'TP': 32, 'TN': 16, 'FP': 7, 'FN': 5}
Learned weights:     [1.048 0.687 0.573]
Learned bias:        0.545


## Step 20 — Predict a New Student's Passing Probability

**What:** Takes a new, unseen student with known features and predicts their probability of passing.

**New student:**
- Study hours = 10.0
- Attendance = 82.0
- Previous score = 67.0

**Critical step:** The new student's features must be standardised using the **training** mean and std — not new statistics. This ensures the input is on the same scale the model was trained on.

**Watch out:** If you forget to scale the new student's features, the prediction will be wrong — the model has no way of knowing the original feature ranges.

In [19]:
new_student = np.array([[10.0, 82.0, 67.0]])
new_student_scaled = (new_student - train_mean) / train_std
new_probability = model.predict_proba(new_student_scaled)[0]

print(f"New student's passing probability: {round(new_probability, 4)}")

New student's passing probability: 0.7654


## Step 21 — Convert the New Student's Probability into a Class

**What:** Applies the threshold (0.5) to the probability to get the final class prediction.

**Why:** The probability alone does not give a decision — the threshold converts it to a binary outcome. The default threshold of 0.5 means: if probability >= 0.5, predict pass (1); otherwise predict fail (0).

**Complete inference path:**

`new features → scale with training stats → linear score → sigmoid → probability → threshold → class`

In [20]:
prediction = model.predict(new_student_scaled)[0]
print(f"Prediction (1=pass, 0=fail): {prediction}")

Prediction (1=pass, 0=fail): 1


---

## Quick Revision — Pipeline at a Glance

| Step | What | Key code |
|---|---|---|
| Sigmoid | Convert score to probability | `1 / (1 + np.exp(-z))` |
| Loss | Measure prediction quality | Binary cross-entropy |
| Training | Learn weights and bias | Gradient descent loop |
| Accuracy | Overall correct predictions | `np.mean(labels == predictions)` |
| Confusion counts | Breakdown of error types | TP, TN, FP, FN |
| Synthetic data | Controlled demonstration dataset | `rng.uniform(...)` |
| Labels | Sample binary outcomes | `rng.binomial(1, probability)` |
| Split | 80/20 train/test | Index permutation |
| Scaling | Normalise features | `(x - train_mean) / train_std` |
| Fit | Learn from training data | `model.fit(features_train_scaled, labels_train)` |
| Predict | Probability + class | `predict_proba()` then `predict()` |
| New student | Inference on unseen input | Scale first, then predict |

## Known issues in the original notebook

| Issue | Location | Effect |
|---|---|---|
| `accuracy()` missing `return` | Step 8 | Prints `None` instead of a number |
| `np.clip(p, 1e-12, 1e-12)` both bounds same | Step 5 | Loss function does not measure correctly |